# V4 + V1 utensil specialist: Simple hybrid comparison

**Inference only. No retraining.** V4 owns food, plate and wrapper; the original frozen V1 specialist owns utensil. Compare each on a few **fresh, unseen cafeteria photographs**.

You need your existing `YOLO_V4_model_results.zip` and original frozen `utensil.pt` from your Mac's local YOLO assets. The latter is **not on GitHub**. T4 GPU is recommended but not required for just a few images.

Output: `*_v4.jpg`, `*_hybrid.jpg` and detection JSON. Visual output does not establish accuracy.


## 1. Install YOLO and load the hybrid script


In [ ]:
import subprocess, sys, zipfile, hashlib, torch
from pathlib import Path
from google.colab import files
print("Installing inference dependencies...",flush=True)
subprocess.run([sys.executable,"-m","pip","install","-q","ultralytics==8.4.175"],check=True)
repo=Path("/content/yolo-cafeteria-research")
if not repo.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/cx-57/yolo-cafeteria-research.git",str(repo)],check=True)
script=repo/"experiments/yolo-v4-4class/v4_hybrid_inference.py"
assert script.exists(), "Hybrid script missing from GitHub clone"
device="0" if torch.cuda.is_available() else "cpu"
print("Inference device:",device,flush=True)


## 2. Upload your **V4 trained results ZIP** (not the dataset ZIP)


In [ ]:
print("Upload YOLO_V4_model_results.zip",flush=True)
uploaded=files.upload()
assert len(uploaded)==1,"Upload one ZIP only"
name,raw=next(iter(uploaded.items()))
assert name.lower().endswith(".zip"),"Select the trained V4 results ZIP"
with zipfile.ZipFile(Path("/content")/name) as z:
    checkpoint=z.read("cafeteria/train/weights/best.pt")
v4=Path("/content/v4_best.pt")
v4.write_bytes(checkpoint)
assert hashlib.sha256(checkpoint).hexdigest()=="0be9895e6c60bfe0b5fd478b01b6aef6484c24f827bf20ea7e65e7e34f66f343","Wrong V4 checkpoint"
print("Verified V4 model",flush=True)


## 3. Upload original **V1 utensil.pt** (from your Mac's local assets folder)


In [ ]:
print("Upload your original frozen V1 utensil.pt",flush=True)
uploaded=files.upload()
assert len(uploaded)==1,"Upload one model file only"
name,raw=next(iter(uploaded.items()))
assert name.lower().endswith(".pt"),"Upload the V1 utensil .pt model"
utensil=Path("/content/utensil_v1.pt")
utensil.write_bytes(raw)
assert hashlib.sha256(raw).hexdigest()=="ce4956547f0c421db1dc4b3d2fa7c28d9d9ae2fc44ddf5acf333ff211c3d4c7f","This is NOT the frozen V1 utensil model"
print("Verified V1 utensil specialist",flush=True)


## 4. Upload a few **new** cafeteria photos (JPG / PNG / WebP)


In [ ]:
print("Upload 3-10 fresh images, not the old train/val/test images",flush=True)
photos=Path("/content/hybrid_input")
photos.mkdir(exist_ok=True)
uploaded=files.upload()
for name,raw in uploaded.items():
    filename=Path(name).name
    assert Path(filename).suffix.lower() in {".jpg",".jpeg",".png",".webp"},"Only image files are supported"
    (photos/filename).write_bytes(raw)
print("Photos:",len(uploaded),flush=True)


## 5. Run paired inference and download the results


In [ ]:
results=Path("/content/YOLO_V4_hybrid_results")
assert not results.exists(),"Results directory exists; use a fresh Colab runtime to avoid overwriting"
subprocess.run([sys.executable,"-u",str(script),"--v4",str(v4),"--utensil",str(utensil),"--images",str(photos),"--output",str(results),"--device",device],check=True)
from IPython.display import Image, display
for image in sorted(results.glob("*_v4.jpg"))[:4]:
    other=results/(image.stem[:-3]+"hybrid.jpg")
    if other.is_file():
        print(image.name,"vs",other.name)
        display(Image(filename=str(image),width=500),Image(filename=str(other),width=500))
archive=Path("/content/YOLO_V4_hybrid_comparison.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for f in results.iterdir():
        z.write(f,arcname=f.name)
files.download(str(archive))
